# 梯度下降

前两章，我们构建了模型的基本框架：

* **前向传播**：根据参数（权重和偏置）计算预测值；
* **损失函数**：量化预测值与真实值之间的误差。

模型参数是我们手动初始化的，并不准确。那么：**如何调整参数，让损失值变小？**

答案是：让网络模型从数据中**主动学习**，这个过程称为**模型训练**（Model Training）。

## 模型训练

模型训练的目标很明确：**找到一组参数（权重和偏置），使损失值尽可能小**。

这是一个经典的数学问题：**函数最优化**。如果把损失值看作一座山，模型训练的目标就是找到山谷（损失值最小的地方）。

神经网络使用的方法叫**梯度下降法**（Gradient Descent）。它的思路非常直观：

> 站在山上，每次都朝着最陡的下坡方向迈一步，最终就能走到山谷。

---

具体到我们的模型，如果把推理函数 `forward()` 代入损失函数 `mse_loss()`，就会发现**损失值 $L$ 是权重 $w$ 和偏置 $b$ 的函数**，记作 $L(w, b)$。

**梯度**（Gradient）就是这个函数"上坡最陡"的方向，用数学语言表达就是偏导数：

$$
\nabla L = \left( \frac{\partial L}{\partial w},\ \frac{\partial L}{\partial b} \right)
$$

**下降**（Descent）就是让参数沿梯度相反的方向移动一步。因为梯度指向上坡，反方向自然就是下坡：

$$
w_{\text{new}} = w_{\text{old}} - \frac{\partial L}{\partial w}
$$
$$
b_{\text{new}} = b_{\text{old}} - \frac{\partial L}{\partial b}
$$

``💡 梯度下降法每次只能保证向局部最优解靠近，不能保证找到全局最优解。但在深度学习的实践中，局部最优解通常已经足够好。``

## 梯度推导

在动手写代码之前，先推导一下梯度的计算公式。

损失值 $L$ 是预测值 $p$ 的函数（损失函数），预测值 $p$ 又是参数 $w$、$b$ 的函数（推理函数）。根据微积分的**链式规则**（Chain Rule），可以把它们的导数"链"起来，得到损失值关于参数的导数：

$$
\frac{\partial L}{\partial w} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial w}
$$

---

**第一步**：计算损失函数关于预测值的导数。

损失函数（单样本）：$L = (y - p)^2$，对 $p$ 求导：

$$
\frac{\partial L}{\partial p} = -2(y - p)
$$

**第二步**：计算推理函数关于参数的导数。

推理函数：$p = w \cdot x + b$，分别对 $w$ 和 $b$ 求导：

$$
\frac{\partial p}{\partial w} = x, \qquad \frac{\partial p}{\partial b} = 1
$$

**第三步**：用链式规则合并，得到损失值关于 $w$ 和 $b$ 的完整梯度：

$$
\frac{\partial L}{\partial w} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial w} = -2(y - p) \cdot x
$$
$$
\frac{\partial L}{\partial b} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial b} = -2(y - p)
$$

---

注意到两个梯度公式都含有同一个公共部分：$-2(y - p)$。在深度学习中，这个公共部分有个专门的名字：**误差项**（Delta），记作 $\delta$：

$$
\delta = -2(y - p)
$$

有了误差项，梯度可以简洁地表示为：

$$
\frac{\partial L}{\partial w} = \delta \cdot x, \qquad \frac{\partial L}{\partial b} = \delta
$$

``💡 误差项 delta 的意义非常直观：预测值偏低（p < y，delta < 0），说明参数需要增大；预测值偏高（p > y，delta > 0），说明参数需要减小。梯度下降正是利用这个信号来纠正参数的。``

In [1]:
import numpy as np

## 数据

继续使用小明的冰激凌店数据：温度 28.1°C、湿度 58%，实际销量 165 个。

### 特征、标签

In [2]:
feature = np.array([28.1, 58.0])
label = np.array([165])

## 模型

在上两章的基础上，本章新增梯度函数和反向函数。

### 参数：权重、偏置

In [3]:
weight = np.ones((1, 2)) / 2
bias = np.zeros(1)

### 推理函数

In [4]:
def forward(x, w, b):
    return x @ w.T + b

### 损失函数（均方误差）

In [5]:
def mse_loss(p, y):
    return np.mean(np.square(y - p))

### 梯度函数

根据上面的推导，梯度函数计算误差项 $\delta$：

$$
\delta = -2(y - p)
$$

In [6]:
def gradient(p, y):
    return -2 * (y - p)

### 反向函数

反向函数利用误差项 $\delta$ 直接更新参数：

$$
w \leftarrow w - \delta \cdot x
$$
$$
b \leftarrow b - \delta
$$

这种数据从输出向输入方向流动的过程，称为**反向传播**（Backpropagation）。

In [7]:
def backward(x, d, w, b):
    w -= d * x
    b -= d
    return w, b

## 验证

在训练之前，先记录当前模型的基准表现。

### 推理

In [8]:
prediction = forward(feature, weight, bias)
print(f'prediction: {prediction}')

prediction: [43.05]


### 评估

In [9]:
loss = mse_loss(prediction, label)
print(f'loss: {loss:.4f}')

loss: 14871.8025


模型预测 43 个，真实值是 165 个，损失值约为 14872。接下来，我们让模型训练一次，看看效果。

## 训练

现在，我们用梯度下降法进行第一次模型训练。

### 梯度计算

计算当前预测值对应的误差项 $\delta$。

In [10]:
delta = gradient(prediction, label)
print(f'delta: {delta}')

delta: [-243.9]


误差项 $\delta$ 为 -243.9，是**负值**。

根据梯度下降的逻辑，参数应该沿梯度**相反**方向更新：负梯度对应的更新方向是**增大参数**。这符合直觉——当前预测值 43 远低于真实值 165，权重和偏置都需要增大。

### 反向传播

In [11]:
weight, bias = backward(feature, delta, weight, bias)
print(f'weight: {weight}')
print(f'bias: {bias}')

weight: [[ 6854.09 14146.7 ]]
bias: [243.9]


权重从 `[0.5, 0.5]` 一步跳到了 `[6854, 14147]`，偏置从 `0` 跳到了 `243.9`。

更新方向是对的：参数确实增大了，符合预期。但增幅如此之大，结果会怎样？

### 重新评估

In [12]:
prediction = forward(feature, weight, bias)
loss = mse_loss(prediction, label)
print(f'new prediction: {prediction}')
print(f'new loss: {loss:.4f}')

new prediction: [1013352.429]
new loss: 1026548766283.6302


损失值从 `14,872` 爆炸到 `1,026,548,766,283`，增大了约 700 亿倍。

发生了什么？

预测值从 `43` 一步跳到了 `1,013,352`，完全越过了真实值 165，一路冲到了山谷的另一边。这就好比下山时，一脚步子太大，直接踩到了对面的山坡上，甚至比出发点更高。

这种现象在深度学习中称为**发散**（Divergence）：损失值不仅没有降低，反而急剧增大，模型训练失败。

怎么办？

## 课后练习

如果预测值比真实值大（预测偏高），误差项 $\delta$ 会是正数还是负数？参数会朝哪个方向更新？